In [50]:
# Created by: CR
# Date: 4/27/26
# Script to run Burnham's Political Debate Model v. 1.1 on data from all states
# Where rescraped data is available, this incorporates the rescraped data without filtering

from transformers import pipeline
import pandas as pd
import os
import numpy as np

### Global Constants; can be changed ###
LABELS = ["economic relief", "reopening", "jobs", "housing", "vaccines",
          "testing", "positive cases", "healthcare professionals", 
          "healthcare infrastructure", "other", "research", "food"]
MULTI_LABEL = True
CHARACTER_NUMBER = 100 #number of characers to slice from each press release
WORKING_DIRECTORY = "/Users/connorrust/Library/CloudStorage/Box-Box/Covid Policies/"
INPUT_DATA = "Data/05_combine_all_states.csv"
OUTPUT_PATH = "Analysis/Testing/Results/06_burnham_all_states.csv"
#####################################
os.chdir(WORKING_DIRECTORY)
data = pd.read_csv(INPUT_DATA)

# defining normalization function
def normalize(matrix, axis=-1):
    """ Takes a numpy 2D array of topic probabilities and normalizes them. 
    (This applies L1 not L2 normalization)
    Args: 
        matrix(numpy array)
        axis(int): axis to normalize across, 1: rows; 0: columns

    Returns:
        2D array with rows/columns normalized

    """
    return matrix / np.sum(matrix, axis=axis, keepdims = True)

# extracting text from data
text = data.pop("Text").str.slice(0,CHARACTER_NUMBER)
lst = text.to_list()[1:5]

ent_template = "This text is about {}"
dis_template = "This text is not about {}"

classes_verbalized = LABELS

zeroshot_classifier = pipeline("zero-shot-classification", 
                               model="mlburnham/Political_DEBATE_DeBERTa_large_v1.1", 
                               device = "mps")  # change the model identifier here

entailment = zeroshot_classifier(lst, classes_verbalized, 
                             hypothesis_template=ent_template, 
                             multi_label=MULTI_LABEL)

disentailment = zeroshot_classifier(lst, classes_verbalized, 
                             hypothesis_template=dis_template, 
                             multi_label=MULTI_LABEL)

clean_output = []

for edct, ddct in zip(entailment, disentailment):
    nd = {}
    nd["sequence"] = edct["sequence"]

    if edct["sequence"] != ddct["sequence"]:
        raise ValueError("entailment and disentailment sequences do not match")
    
    for idx, label in enumerate(edct["labels"]):
        nd[label + "_ent"] = edct["scores"][idx]
    for idx, label in enumerate(ddct["labels"]):
        nd[label + "_dis"] = ddct["scores"][idx]

    clean_output.append(nd)

df = pd.DataFrame(clean_output)


##### FIX THIS TO DEAL WITH HAVING BOTH SETS OF COLUMNS
# separating out only disentailment and entailment scores into separate dfs
ent_df = df.filter(like = "ent")
dis_df = df.filter(like = "dis")
 
# copies to normalize
ent_df2 = ent_df.copy(deep=False) 
dis_df2 = dis_df.copy(deep=False) 

# L1 Normalizing each row
ent_df2.loc[:,ent_df2.columns.str.contains("ent")] = normalize(ent_df.values, axis=1)
dis_df2.loc[:,dis_df2.columns.str.contains("dis")] = normalize(dis_df.values, axis=1)

# combining probabilities with the original data
output = pd.concat([data, ent_df2, dis_df2], axis=1)




Loading weights: 100%|██████████| 394/394 [00:00<00:00, 7651.96it/s]


In [51]:
output

,Title,State,Agency,Date,economic relief_ent,reopening_ent,positive cases_ent,other_ent,testing_ent,food_ent,...,vaccines_dis,housing_dis,healthcare infrastructure_dis,jobs_dis,food_dis,reopening_dis,testing_dis,other_dis,positive cases_dis,economic relief_dis
0,10 California Communities Awarded $17 Million ...,CA,Governor,2022-06-24,9.999425e-01,0.000026,2.089037e-05,0.000005,0.000002,9.381461e-07,...,0.092251,0.092251,0.092251,0.092250,0.092250,0.092249,9.224893e-02,0.092248,0.077501,1.582849e-07
1,12:45 PM: Governor Newsom to Make Major Announ...,CA,Governor,2020-03-25,1.890901e-02,0.246505,1.945257e-02,0.552035,0.018756,1.841187e-02,...,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,8.333336e-02,0.083333,0.083333,8.333336e-02
2,2021 Media Credentialing Information,CA,Governor,2021-02-08,2.318982e-03,0.010974,2.502962e-03,0.621509,0.002131,1.891711e-03,...,0.083337,0.083298,0.083337,0.083337,0.083337,0.083337,8.333662e-02,0.083336,0.083337,8.333661e-02
3,$240 Million Available to Resolve Encampments ...,CA,Governor,2022-12-01,1.668169e-07,0.000241,4.895174e-07,0.000016,0.995566,1.814947e-07,...,0.099986,0.099986,0.000150,0.099986,0.099986,0.099977,3.024475e-07,0.099986,0.099986,9.998588e-02
4,3:30 PM: Governor Newsom to Provide Update on ...,CA,Governor,2020-03-23,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15879,UVA Health Scheduling COVID Vaccine Appointmen...,VA,University,2021-05-12,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15880,UVA Health to Require COVID-19 Vaccination for...,VA,University,2021-08-25,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15881,UVA Health Joins National Trial Testing Medica...,VA,University,2022-02-21,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15882,"UVA Tests Drug, Given to Trump, to See If It C...",VA,University,2020-10-08,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [49]:
ent_df2.loc[:,ent_df2.columns.str.contains("ent")]

,economic relief_ent,reopening_ent,positive cases_ent,other_ent,testing_ent,food_ent,healthcare infrastructure_ent,housing_ent,jobs_ent,vaccines_ent,healthcare professionals_ent,research_ent
0,9.995472e-01,2.637096e-05,2.088211e-05,0.000005,2.408934e-06,9.377753e-07,6.468366e-07,4.892846e-07,4.647899e-07,3.324896e-07,1.985342e-07,1.582566e-07
1,1.615965e-07,2.106630e-06,1.662418e-07,0.000005,1.602867e-07,1.573480e-07,1.701422e-07,1.617073e-07,2.534268e-07,1.649744e-07,1.525641e-07,1.733946e-07
2,1.921027e-07,9.090522e-07,2.073435e-07,0.000051,1.765373e-07,1.567079e-07,1.617040e-07,2.888418e-05,1.638895e-07,1.695750e-07,1.718489e-07,1.609895e-07
3,1.675590e-07,2.418915e-04,4.916950e-07,0.000016,9.999943e-01,1.823021e-07,4.189932e-03,1.710777e-07,1.696694e-07,5.780639e-07,3.865493e-06,2.059711e-07


In [35]:
ent_df2


array([[9.99942454e-01, 2.63813837e-05, 2.08903653e-05, 4.63494642e-06,
        2.40988689e-06, 9.38146112e-07, 6.47092345e-07, 4.89478104e-07,
        4.64973707e-07, 3.32621060e-07, 1.98612691e-07, 1.58319137e-07],
       [1.89090069e-02, 2.46504611e-01, 1.94525691e-02, 5.52034960e-01,
        1.87557373e-02, 1.84118685e-02, 1.99089656e-02, 1.89219673e-02,
        2.96544122e-02, 1.93042698e-02, 1.78520867e-02, 2.02895455e-02],
       [2.31898179e-03, 1.09736912e-02, 2.50296242e-03, 6.21508710e-01,
        2.13108284e-03, 1.89171058e-03, 1.95202133e-03, 3.48677517e-01,
        1.97840377e-03, 2.04703680e-03, 2.07448661e-03, 1.94339627e-03],
       [1.66816914e-07, 2.40820228e-04, 4.89517400e-07, 1.64059168e-05,
        9.95565592e-01, 1.81494714e-07, 4.17137600e-03, 1.70320052e-07,
        1.68917993e-07, 5.75503806e-07, 3.84837362e-06, 2.05058906e-07]])

In [12]:
df3

,sequence,economic relief,reopening,positive cases,other,testing,food,healthcare infrastructure,housing,jobs,vaccines,healthcare professionals,research
0,SACRAMENTO – Governor Gavin Newsom will today ...,9.995472e-01,2.637096e-05,2.088211e-05,0.000005,2.408934e-06,9.377753e-07,6.468366e-07,4.892846e-07,4.647899e-07,3.324896e-07,1.985342e-07,1.582566e-07
1,Information regarding 2021 Governor’s Office m...,1.615965e-07,2.106630e-06,1.662418e-07,0.000005,1.602867e-07,1.573480e-07,1.701422e-07,1.617073e-07,2.534268e-07,1.649744e-07,1.525641e-07,1.733946e-07
2,SACRAMENTO – California is cleaning up encampm...,1.921027e-07,9.090522e-07,2.073435e-07,0.000051,1.765373e-07,1.567079e-07,1.617040e-07,2.888418e-05,1.638895e-07,1.695750e-07,1.718489e-07,1.609895e-07
3,SACRAMENTO – Governor Gavin Newsom and state h...,1.675590e-07,2.418915e-04,4.916950e-07,0.000016,9.999943e-01,1.823021e-07,4.189932e-03,1.710777e-07,1.696694e-07,5.780639e-07,3.865493e-06,2.059711e-07


In [4]:
df1 = pd.DataFrame(entailment[1])
df1.pivot(index = "sequence", columns = "labels", values = "scores")

labels,economic relief,food,healthcare infrastructure,healthcare professionals,housing,jobs,other,positive cases,reopening,research,testing,vaccines
sequence,,,,,,,,,,,,
Information regarding 2021 Governor’s Office media credentials: Working members of the media who reg,1.615965e-07,1.573480e-07,1.701422e-07,1.525641e-07,1.617073e-07,2.534268e-07,0.000005,1.662418e-07,0.000002,1.733946e-07,1.602867e-07,1.649744e-07


In [33]:
prob_cols = df.columns.difference(['sequence'])
prob_cols
# creating a copy to normalize
df2 = df.copy(deep=False) 
# Normalizing each row
df2[prob_cols] = normalize(df[prob_cols].values, axis=1)
df2
output =pd.concat([data, df2], axis=1)
output
df2
type(prob_cols)

df[prob_cols]
df.filter(like='ent')
df
df.filter(like='dis')
df
df.filter(like='dis').values


array([[9.99999821e-01, 9.99999821e-01, 9.99999702e-01, 9.99999642e-01,
        9.99999404e-01, 9.99999106e-01, 9.99995887e-01, 9.99985099e-01,
        9.99982297e-01, 9.99969602e-01, 8.40110242e-01, 1.71581542e-06],
       [9.99999702e-01, 9.99999821e-01, 9.99999821e-01, 9.99999821e-01,
        9.99999821e-01, 9.99999464e-01, 9.99999762e-01, 9.99999583e-01,
        9.99999821e-01, 9.99996662e-01, 9.99999821e-01, 9.99999821e-01],
       [9.99999821e-01, 9.99999821e-01, 9.99999821e-01, 9.99533117e-01,
        9.99999821e-01, 9.99999821e-01, 9.99999821e-01, 9.99999642e-01,
        9.99999821e-01, 9.99993861e-01, 9.99999821e-01, 9.99999702e-01],
       [9.99999702e-01, 9.99999285e-01, 9.99999702e-01, 9.99999821e-01,
        1.49624853e-03, 9.99999821e-01, 9.99999821e-01, 9.99915600e-01,
        3.02490116e-06, 9.99998331e-01, 9.99999404e-01, 9.99999821e-01]])

In [16]:
dict = {}
dict["rice"] = "chicken"


KeyError: 'chicken'

In [ ]:


### Global Constants; can be changed ###
LABELS = ["economic relief", "reopening", "jobs", "housing", "vaccines",
          "testing", "positive cases", "healthcare professionals", 
          "healthcare infrastructure", "other", "research", "food"]
MULTI_LABEL = True
CHARACTER_NUMBER = 100 #number of characers to slice from each press release
WORKING_DIRECTORY = "/Users/connorrust/Library/CloudStorage/Box-Box/Covid Policies/"
INPUT_DATA = "Data/05_combine_all_states.csv"
OUTPUT_PATH = "Analysis/Testing/Results/06_burnham_all_states.csv"
#####################################
os.chdir(WORKING_DIRECTORY)
data = pd.read_csv(INPUT_DATA)

# defining normalization function
def normalize(matrix, axis=-1):
    """ Takes a numpy 2D array of topic probabilities and normalizes them. 
    (This applies L1 not L2 normalization)
    Args: 
        matrix(numpy array)
        axis(int): axis to normalize across, 1: rows; 0: columns

    Returns:
        2D array with rows/columns normalized

    """
    return matrix / np.sum(matrix, axis=axis, keepdims = True)

# extracting text from data
text = data.pop("Text").str.slice(0,CHARACTER_NUMBER)
lst = text.to_list()[1:5]

ent_template = "This text is about {}"
dis_template = "This text is not about {}"

classes_verbalized = LABELS

zeroshot_classifier = pipeline("zero-shot-classification", 
                               model="mlburnham/Political_DEBATE_DeBERTa_large_v1.1", 
                               device = "mps")  # change the model identifier here

entailment = zeroshot_classifier(lst, classes_verbalized, 
                             hypothesis_template=ent_template, 
                             multi_label=MULTI_LABEL)

disentailment = zeroshot_classifier(lst, classes_verbalized, 
                             hypothesis_template=dis_template, 
                             multi_label=MULTI_LABEL)

clean_output = []

for edct, ddct in zip(entailment, disentailment):
    nd = {}
    nd["sequence"] = edct["sequence"]

    if edct["sequence"] != ddct["sequence"]:
        raise ValueError("entailment and disentailment sequences do not match")
    
    for idx, label in enumerate(edct["labels"]):
        nd[label + "_ent"] = edct["scores"][idx]
    for idx, label in enumerate(ddct["labels"]):
        nd[label + "_dis"] = ddct["scores"][idx]
        
    clean_output.append(nd)

df = pd.DataFrame(clean_output)

Loading weights: 100%|██████████| 394/394 [00:00<00:00, 9788.57it/s]


In [7]:
df

,sequence,economic relief_ent,reopening_ent,positive cases_ent,other_ent,testing_ent,food_ent,healthcare infrastructure_ent,housing_ent,jobs_ent,...,vaccines_dis,housing_dis,healthcare infrastructure_dis,jobs_dis,food_dis,reopening_dis,testing_dis,other_dis,positive cases_dis,economic relief_dis
0,SACRAMENTO – Governor Gavin Newsom will today ...,9.995472e-01,2.637096e-05,2.088211e-05,0.000005,2.408934e-06,9.377753e-07,6.468366e-07,4.892846e-07,4.647899e-07,...,1.0,1.000000,0.999999,0.999999,0.999996,0.999985,0.999982,0.999970,0.840110,0.000002
1,Information regarding 2021 Governor’s Office m...,1.615965e-07,2.106630e-06,1.662418e-07,0.000005,1.602867e-07,1.573480e-07,1.701422e-07,1.617073e-07,2.534268e-07,...,1.0,1.000000,1.000000,0.999999,1.000000,1.000000,1.000000,0.999997,1.000000,1.000000
2,SACRAMENTO – California is cleaning up encampm...,1.921027e-07,9.090522e-07,2.073435e-07,0.000051,1.765373e-07,1.567079e-07,1.617040e-07,2.888418e-05,1.638895e-07,...,1.0,0.999533,1.000000,1.000000,1.000000,1.000000,1.000000,0.999994,1.000000,1.000000
3,SACRAMENTO – Governor Gavin Newsom and state h...,1.675590e-07,2.418915e-04,4.916950e-07,0.000016,9.999943e-01,1.823021e-07,4.189932e-03,1.710777e-07,1.696694e-07,...,1.0,1.000000,0.001496,1.000000,1.000000,0.999916,0.000003,0.999998,0.999999,1.000000
